In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q timm


In [ ]:
import os, torch, timm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import label_binarize

In [ ]:
from google.colab import files
files.upload()

import os
os.makedirs('/root/.config/kaggle', exist_ok=True)
!cp kaggle.json /root/.config/kaggle/
!chmod 600 /root/.config/kaggle/kaggle.json

os.makedirs('/content/drive/MyDrive/HAM10000', exist_ok=True)

!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 \
    -p /content/drive/MyDrive/HAM10000

!unzip -q /content/drive/MyDrive/HAM10000/skin-cancer-mnist-ham10000.zip \
    -d /content/drive/MyDrive/HAM10000

!ls /content/drive/MyDrive/HAM10000


KeyboardInterrupt: 

In [ ]:
!ls /content/drive/MyDrive/HAM10000

HAM10000_images_part_1	plots
HAM10000_images_part_2	skin-cancer-mnist-ham10000.zip


In [ ]:
!ls /content/drive/MyDrive/HAM10000/HAM10000_images_part_1 | head -5
!find /content/drive/MyDrive/HAM10000 -name "*.csv"

ISIC_0024306.jpg
ISIC_0024307.jpg
ISIC_0024308.jpg
ISIC_0024309.jpg
ISIC_0024310.jpg


In [ ]:
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 \
    -p /content/drive/MyDrive/HAM10000 --file HAM10000_metadata.csv

!ls /content/drive/MyDrive/HAM10000/*.csv

Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
100% 550k/550k [00:00<00:00, 1.77MB/s]

/content/drive/MyDrive/HAM10000/HAM10000_metadata.csv


In [ ]:
BASE_DIR  = '/content/drive/MyDrive/HAM10000'
IMG_DIR1 = f'{BASE_DIR}/HAM10000_images_part_1'
IMG_DIR2 = f'{BASE_DIR}/HAM10000_images_part_2'
CSV_PATH  = f'{BASE_DIR}/HAM10000_metadata.csv'
CKPT_PATH = f'{BASE_DIR}/best_model.pt'
PLOT_DIR  = f'{BASE_DIR}/plots'
os.makedirs(PLOT_DIR, exist_ok=True)

CLASSES  = ['akiec','bcc','bkl','df','mel','nv','vasc']
NUM_CLS  = len(CLASSES)
IMG_SIZE = 224
BATCH    = 32
EPOCHS   = 30
LR       = 1e-4
WD       = 1e-4   # weight decay
PATIENCE = 6      # early stopping patience
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(DEVICE)

cuda


In [ ]:
df = pd.read_csv(CSV_PATH)
df['label'] = df['dx'].map({c: i for i, c in enumerate(CLASSES)})

def find_img(img_id):
    for d in [IMG_DIR1, IMG_DIR2]:
        p = f'{d}/{img_id}.jpg'
        if os.path.exists(p): return p
    return None

df['path'] = df['image_id'].apply(find_img)
df = df.dropna(subset=['path'])
print(df['dx'].value_counts())

dx
nv       5836
bkl       923
mel       763
bcc       459
akiec     310
vasc      115
df         98
Name: count, dtype: int64


In [ ]:
counts = df['label'].value_counts().sort_index().values

# Loss weights — inverse frequency
class_weights = torch.tensor(1.0 / counts, dtype=torch.float32)
class_weights = (class_weights / class_weights.sum() * NUM_CLS).to(DEVICE)

# Sampler weights — one per sample (fixes imbalanced batches)
sample_weights = torch.tensor([1.0 / counts[l] for l in df['label']], dtype=torch.float32)

print('Class weights:')
for c, w in zip(CLASSES, class_weights.cpu()): print(f'  {c}: {w:.4f}')

Class weights:
  akiec: 0.8404
  bcc: 0.5676
  bkl: 0.2823
  df: 2.6583
  mel: 0.3414
  nv: 0.0446
  vasc: 2.2654


In [ ]:
print(df.columns.tolist())
print(df.head(3))
print(df['dx'].unique())

['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization', 'label', 'path']
Empty DataFrame
Columns: [lesion_id, image_id, dx, dx_type, age, sex, localization, label, path]
Index: []
[]


In [ ]:
!ls /content/drive/MyDrive/HAM10000/HAM10000_images_part_1 | head -3
print(pd.read_csv(CSV_PATH)['image_id'].iloc[0])

ISIC_0024306.jpg
ISIC_0024307.jpg
ISIC_0024308.jpg
ISIC_0027419


In [ ]:
import os
path = f'/content/drive/MyDrive/HAM10000/HAM10000_images_part_1/ISIC_0024306.jpg'
print(os.path.exists(path))

True


In [ ]:
df_raw = pd.read_csv(CSV_PATH)
sample_id = df_raw['image_id'].iloc[0]
print(f"Sample image_id: '{sample_id}'")

p1 = f'{IMG_DIR1}/{sample_id}.jpg'
p2 = f'{IMG_DIR2}/{sample_id}.jpg'
print(f"Path1 exists: {os.path.exists(p1)} → {p1}")
print(f"Path2 exists: {os.path.exists(p2)} → {p2}")

Sample image_id: 'ISIC_0027419'
Path1 exists: False → /content/drive/MyDrive/HAM10000/ham10000_images_part_1/ISIC_0027419.jpg
Path2 exists: False → /content/drive/MyDrive/HAM10000/ham10000_images_part_2/ISIC_0027419.jpg


In [ ]:
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
print(f'Train: {len(train_df)}  Val: {len(val_df)}')

train_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    transforms.RandomErasing(p=0.2),   # Cutout regularisation
])

val_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

class SkinDataset(Dataset):
    def __init__(self, df, tfm):
        self.df  = df.reset_index(drop=True)
        self.tfm = tfm
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        try:
            img = Image.open(row['path']).convert('RGB')
        except Exception:
            # return a blank image if file is corrupted
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (0, 0, 0))
        return self.tfm(img), int(row['label'])

Train: 6803  Val: 1701


In [ ]:
class EfficientNetWithDropout(nn.Module):
    def __init__(self, num_classes, drop_rate=0.4):
        super().__init__()
        self.base = timm.create_model('efficientnet_b3', pretrained=True, num_classes=0)
        feat_dim  = self.base.num_features          # 1536 for B3
        self.head = nn.Sequential(
            nn.BatchNorm1d(feat_dim),
            nn.Dropout(drop_rate),
            nn.Linear(feat_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(drop_rate / 2),
            nn.Linear(512, num_classes),
        )
    def forward(self, x):
        return self.head(self.base(x))

model     = EfficientNetWithDropout(NUM_CLS).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

In [ ]:
!pip install -q tqdm

from tqdm import tqdm

def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss = 0
    all_probs, all_labels = [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    desc = 'Train' if training else 'Val'

    with ctx:
        pbar = tqdm(loader, desc=desc, leave=False)
        for imgs, labels in pbar:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if training: optimizer.zero_grad()

            logits = model(imgs)
            loss   = criterion(logits, labels)

            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item()
            all_probs.append(torch.softmax(logits, dim=1).detach().cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    lb  = label_binarize(all_labels, classes=list(range(NUM_CLS)))
    auc = roc_auc_score(lb, all_probs, average='macro', multi_class='ovr')
    return total_loss / len(loader), auc



In [ ]:
history = {'train_loss':[], 'val_loss':[], 'train_auc':[], 'val_auc':[], 'lr':[]}
best_auc     = 0.0
patience_ctr = 0

for epoch in range(1, EPOCHS+1):
    print(f'\n── Epoch {epoch:02d}/{EPOCHS} ──')
    tr_loss, tr_auc = run_epoch(train_loader, training=True)
    vl_loss, vl_auc = run_epoch(val_loader,   training=False)
    scheduler.step()
    cur_lr = scheduler.get_last_lr()[0]

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_auc'].append(tr_auc)
    history['val_auc'].append(vl_auc)
    history['lr'].append(cur_lr)

    print(f'  Train  → Loss: {tr_loss:.4f}  AUC: {tr_auc:.4f}')
    print(f'  Val    → Loss: {vl_loss:.4f}  AUC: {vl_auc:.4f}')
    print(f'  LR     → {cur_lr:.2e}')

    if vl_auc > best_auc:
        best_auc     = vl_auc
        patience_ctr = 0
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'auc': vl_auc, 'history': history}, CKPT_PATH)
        print(f'  ✓ Checkpoint saved  (best AUC {best_auc:.4f})')
    else:
        patience_ctr += 1
        print(f'  ✗ No improvement    ({patience_ctr}/{PATIENCE})')
        if patience_ctr >= PATIENCE:
            print(f'\n⚑ Early stopping triggered at epoch {epoch}')
            break

print(f'\n✓ Training complete — Best Val AUC: {best_auc:.4f}')


── Epoch 01/30 ──


  Train  → Loss: 2.2224  AUC: 0.8983
  Val    → Loss: 2.1100  AUC: 0.9146
  LR     → 9.76e-05
  ✓ Checkpoint saved  (best AUC 0.9146)

── Epoch 02/30 ──


  Train  → Loss: 2.0797  AUC: 0.9064
  Val    → Loss: 2.0477  AUC: 0.9320
  LR     → 9.57e-05
  ✓ Checkpoint saved  (best AUC 0.9320)

── Epoch 03/30 ──


Train:  57%|█████▋    | 122/213 [03:32<01:11,  1.26it/s, loss=3.2158]

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('EfficientNet-B3 — HAM10000 Training', fontsize=14)

# Loss
axes[0].plot(epochs_ran, history['train_loss'], label='Train loss', marker='o', ms=4)
axes[0].plot(epochs_ran, history['val_loss'],   label='Val loss',   marker='o', ms=4)
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('CrossEntropy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# AUC
axes[1].plot(epochs_ran, history['train_auc'], label='Train AUC', marker='o', ms=4)
axes[1].plot(epochs_ran, history['val_auc'],   label='Val AUC',   marker='o', ms=4)
axes[1].axhline(best_auc, color='green', linestyle='--', linewidth=1, label=f'Best {best_auc:.4f}')
axes[1].set_title('Macro AUC-ROC')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# LR
axes[2].plot(epochs_ran, history['lr'], color='orange', marker='o', ms=4)
axes[2].set_title('Learning rate schedule')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('LR')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/training_curves.png', dpi=150)
plt.show()

In [ ]:
ckpt = torch.load(CKPT_PATH)
model.load_state_dict(ckpt['model'])
print(f'Loaded epoch {ckpt["epoch"]} — AUC {ckpt["auc"]:.4f}\n')

model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        probs = torch.softmax(model(imgs.to(DEVICE)), dim=1).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())

all_probs  = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)
preds      = all_probs.argmax(axis=1)
lb         = label_binarize(all_labels, classes=list(range(NUM_CLS)))

# Per-class AUC
print('Per-class AUC:')
for i, cls in enumerate(CLASSES):
    auc = roc_auc_score(lb[:, i], all_probs[:, i])
    print(f'  {cls:6s}: {auc:.4f}')
print(f'\nMacro AUC: {roc_auc_score(lb, all_probs, average="macro", multi_class="ovr"):.4f}')

# Confusion matrix
fig, ax = plt.subplots(figsize=(9, 7))
cm = confusion_matrix(all_labels, preds)
disp = ConfusionMatrixDisplay(cm, display_labels=CLASSES)
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title('Confusion Matrix — Validation Set')
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/confusion_matrix.png', dpi=150)
plt.show()